In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical

In [2]:
data = pd.read_csv('/content/judge-1377884607_tweet_product_company.csv', encoding='latin-1')

In [3]:
data.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [4]:
# Selecting only the 'tweet_text' and 'is_there_an_emotion_directed_at_a_brand_or_product' columns for sentiment analysis

data=data[['is_there_an_emotion_directed_at_a_brand_or_product']]

In [5]:
# Rename columns for simplicity

data.columns = ['tweet', 'sentiment']

In [6]:
data.shape

(9093, 2)

In [7]:
pd.set_option('display.max_colwidth',None)

In [8]:
data.head()

,tweet,sentiment
0,".@wesley83 I have a 3G iPhone. After 3 hrs tweeting at #RISE_Austin, it was dead! I need to upgrade. Plugin stations at #SXSW.",Negative emotion
1,"@jessedee Know about @fludapp ? Awesome iPad/iPhone app that you'll likely appreciate for its design. Also, they're giving free Ts at #SXSW",Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. They should sale them down at #SXSW.,Positive emotion
3,@sxsw I hope this year's festival isn't as crashy as this year's iPhone app. #sxsw,Negative emotion
4,"@sxtxstate great stuff on Fri #SXSW: Marissa Mayer (Google), Tim O'Reilly (tech books/conferences) &amp; Matt Mullenweg (Wordpress)",Positive emotion


In [9]:
data['sentiment'].value_counts()

,count
sentiment,
No emotion toward brand or product,5389
Positive emotion,2978
Negative emotion,570
I can't tell,156



 data preprocessing

In [10]:
data['tweet'].isnull().sum()

1

In [11]:
data['tweet'].fillna('', inplace=True)

<ipython-input-11-0b7cac3416bf>:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['tweet'].fillna('', inplace=True)


feature extraction

In [13]:
x = data['tweet']
y = data['sentiment']

In [14]:
y

,sentiment
0,Negative emotion
1,Positive emotion
2,Positive emotion
3,Negative emotion
4,Positive emotion
...,...
9088,Positive emotion
9089,No emotion toward brand or product
9090,No emotion toward brand or product
9091,No emotion toward brand or product


 encoding

In [15]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)
y = to_categorical(y)

In [16]:
y

array([[0., 1., 0., 0.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.],
       ...,
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.]])

In [17]:
# Tokenize the text data using Keras Tokenizer

In [20]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data['tweet'])
tokenized_text = tokenizer.texts_to_sequences(data['tweet'])

In [21]:
len(tokenized_text[0])

24

In [22]:
len(tokenized_text[1])

22

Pad the tokenized_text to make all text sequences the same length (100)

In [23]:
from keras.utils import pad_sequences
x=pad_sequences(tokenized_text,maxlen=100)

train_test_split

In [24]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

SimpleRNN

In [25]:
from keras.models import Sequential
from keras.layers import Dense,LSTM,Embedding,SimpleRNN,Dropout

In [26]:
len(tokenizer.word_index)

10147

In [27]:
# Create a Sequential model
model = Sequential()

# Add an Embedding layer
model.add(Embedding(input_dim = len(tokenizer.word_index)+1, output_dim=128, input_length=100))

# Add a SimpleRNN layer with 32 units

model.add(SimpleRNN(32))
model.add(Dropout(0.5))

# Add a Dense layer with 50 units
#model.add(Dense(50,activation = 'relu'))
#model.add(Dropout(0.5))

# Add the final Dense layer with 4 units (for 4 classes)
model.add(Dense(4, activation='softmax'))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [28]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [29]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [30]:
model.fit(x_train,y_train,epochs=10,validation_split=0.1)

Epoch 1/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 11s 37ms/step - accuracy: 0.5135 - loss: 1.0433 - val_accuracy: 0.6140 - val_loss: 0.8610
Epoch 2/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 11s 40ms/step - accuracy: 0.7094 - loss: 0.7489 - val_accuracy: 0.6168 - val_loss: 0.8952
Epoch 3/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.8540 - loss: 0.4490 - val_accuracy: 0.5755 - val_loss: 1.0594
Epoch 4/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 8s 38ms/step - accuracy: 0.9036 - loss: 0.3033 - val_accuracy: 0.5920 - val_loss: 1.1485
Epoch 5/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 9s 43ms/step - accuracy: 0.9227 - loss: 0.2358 - val_accuracy: 0.5920 - val_loss: 1.2412
Epoch 6/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - accuracy: 0.9319 - loss: 0.1993 - val_accuracy: 0.6181 - val_loss: 1.2090
Epoch 7/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 9s 39ms/step - accuracy: 0.9362 - loss: 0.1771 - val_accuracy: 0.5920 - val_loss: 1.3064
Epoch 8/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.9334 - loss: 0.1791 - val_

In [31]:
y_pred=model.predict(x_test)

57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step


In [32]:
accuracy = model.evaluate(x_test, y_test)[1]
print(f'Test Accuracy: {accuracy * 100:.2f}%')

57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5942 - loss: 1.3468
Test Accuracy: 59.92%


LSTM

In [33]:
model = Sequential()
model.add(Embedding(input_dim = len(tokenizer.word_index)+1, output_dim=128, input_length=100))
model.add(LSTM(32))
model.add(Dense(4, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train,y_train,epochs=10,validation_split=0.1)

Epoch 1/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - accuracy: 0.5875 - loss: 1.0010 - val_accuracy: 0.6264 - val_loss: 0.8241
Epoch 2/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 20s 66ms/step - accuracy: 0.7071 - loss: 0.7282 - val_accuracy: 0.6566 - val_loss: 0.7935
Epoch 3/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 21s 66ms/step - accuracy: 0.8226 - loss: 0.4852 - val_accuracy: 0.6607 - val_loss: 0.8255
Epoch 4/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 20s 66ms/step - accuracy: 0.8712 - loss: 0.3551 - val_accuracy: 0.6470 - val_loss: 0.9120
Epoch 5/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 13s 65ms/step - accuracy: 0.8982 - loss: 0.2817 - val_accuracy: 0.6676 - val_loss: 1.0180
Epoch 6/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 20s 65ms/step - accuracy: 0.9112 - loss: 0.2398 - val_accuracy: 0.6401 - val_loss: 1.0763
Epoch 7/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 14s 66ms/step - accuracy: 0.9152 - loss: 0.2008 - val_accuracy: 0.6593 - val_loss: 1.2182
Epoch 8/10
205/205 ━━━━━━━━━━━━━━━━━━━━ 14s 67ms/step - accuracy: 0.9328 - loss: 0.1665 - 

In [34]:
y_pred=model.predict(x_test)

57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step


In [35]:
accuracy = model.evaluate(x_test, y_test)[1]
print(f'Test Accuracy: {accuracy * 100:.2f}%')

57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6575 - loss: 1.5020
Test Accuracy: 65.31%
